# Lab 5: Fault Slip Rates from Displaced Landforms — Wallace Creek

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/05_wallace_creek.ipynb)

## Introduction

Wallace Creek is one of the most clearly documented strike-slip fault offset sites in the world. The creek, named after USGS geologist Robert Wallace who described it in the 1960s, crosses the central San Andreas fault in the Carrizo Plain of central California. Its channel has been repeatedly deflected by right-lateral fault slip, creating a series of abandoned meanders that record the cumulative displacement history of the fault.

In this lab you will work with a 0.5 m resolution LiDAR digital elevation model (DEM) of the site to:
1. Identify the fault trace and its geomorphic expression
2. Measure the right-lateral offset of three generations of Wallace Creek channels
3. Interpret the landscape history using radiocarbon ages from Sieh and Jahns (1984)
4. Calculate slip rates for the San Andreas fault with formal uncertainty estimates

## Learning objectives

By the end of this lab you will be able to:
- Generate and interpret hillshade and slope maps from a LiDAR DEM
- Identify fault-related landforms including scarps, sag ponds, and deflected drainages
- Project channel walls to a fault trace to measure fault-parallel offset
- Propagate uncertainty through a slip-rate calculation using Monte Carlo methods
- Evaluate the assumptions embedded in a geomorphic slip-rate estimate

## Notebook outline

1. [Setup](#1-setup)
2. [Load and visualize the DEM](#2-load-and-visualize-the-dem)
3. [Define the fault coordinate system](#3-define-the-fault-coordinate-system)
4. [Measure channel offsets](#4-measure-channel-offsets)
5. [Interpret the geomorphic history](#5-interpret-the-geomorphic-history)
6. [Calculate slip rates](#6-calculate-slip-rates)
7. [Synthesis](#synthesis)

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import uniform_filter
from urllib.request import urlopen
import io

# ── Download the LiDAR DEM ────────────────────────────────────────────────────
DATA_URL = (
    "https://raw.githubusercontent.com/amtseismo/EPS166/main/"
    "datasets/WallaceCreek/WallaceCreek.asc"
)

print(f"Downloading DEM from:\n  {DATA_URL}")
with urlopen(DATA_URL, timeout=120) as r:
    raw = r.read().decode()
print("Download complete.")

In [ ]:
# ── Parse the ESRI ASCII raster ───────────────────────────────────────────────
lines = raw.strip().split("\n")
header = {}
data_start = 0
for i, line in enumerate(lines):
    parts = line.strip().split()
    if parts[0].lower() in ("ncols","nrows","xllcorner","yllcorner","cellsize","nodata_value"):
        header[parts[0].lower()] = float(parts[1])
        data_start = i + 1
    else:
        break

NCOLS  = int(header["ncols"])
NROWS  = int(header["nrows"])
XLL    = header["xllcorner"]     # Easting of left edge (UTM Zone 11, m)
YLL    = header["yllcorner"]     # Northing of bottom edge
CELL   = header["cellsize"]      # 0.5 m
NODATA = header["nodata_value"]

dem = np.array(
    [list(map(float, lines[data_start + i].split())) for i in range(NROWS)],
    dtype=np.float32
)
dem[dem == NODATA] = np.nan

# Coordinate arrays (cell centres)
east   = XLL + (np.arange(NCOLS) + 0.5) * CELL   # shape (ncols,)
north  = YLL + (NROWS - np.arange(NROWS) - 0.5) * CELL  # shape (nrows,), N at top

print(f"DEM: {NCOLS} x {NROWS} cells at {CELL} m resolution")
print(f"Easting:  {XLL:.0f} – {XLL+NCOLS*CELL:.0f} m")
print(f"Northing: {YLL:.0f} – {YLL+NROWS*CELL:.0f} m  (UTM Zone 11)")
print(f"Elevation: {np.nanmin(dem):.1f} – {np.nanmax(dem):.1f} m")

## 2. Load and visualize the DEM

We compute a hillshade — a synthetic image of the terrain illuminated from a given sun direction — to reveal subtle topographic features that are hard to see in raw elevation data.

In [ ]:
def hillshade(dem, cell, azimuth_deg=315, altitude_deg=45):
    """Compute a greyscale hillshade from a DEM.
    azimuth_deg : sun direction (clockwise from N, default NW = 315°)
    altitude_deg: sun angle above horizon (default 45°)
    """
    dx, dy = np.gradient(dem, cell, cell)
    slope  = np.arctan(np.sqrt(dx**2 + dy**2))
    aspect = np.arctan2(-dy, dx)               # counter-clockwise from E
    az  = np.deg2rad(azimuth_deg)
    alt = np.deg2rad(altitude_deg)
    hs  = (np.cos(alt) * np.cos(slope)
           + np.sin(alt) * np.sin(slope) * np.cos(az - aspect))
    return np.clip(hs, 0, 1)

hs = hillshade(dem, CELL, azimuth_deg=315, altitude_deg=45)

# Extent for imshow: [x_left, x_right, y_bottom, y_top]
EXT = [XLL, XLL + NCOLS*CELL, YLL, YLL + NROWS*CELL]

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(hs, cmap="gray", extent=EXT, origin="upper", vmin=0, vmax=1)
ax.set_xlabel("Easting (m, UTM Zone 11)", fontsize=11)
ax.set_ylabel("Northing (m)", fontsize=11)
ax.set_title("Wallace Creek LiDAR — hillshade (315°/45°)", fontsize=12)

# Scale bar
ax.plot([XLL+50, XLL+150], [YLL+30, YLL+30], "k-", lw=3)
ax.text(XLL+100, YLL+50, "100 m", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Slope map ─────────────────────────────────────────────────────────────────
dx, dy = np.gradient(dem, CELL, CELL)
slope_deg = np.rad2deg(np.arctan(np.sqrt(dx**2 + dy**2)))

fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
axes[0].imshow(hs, cmap="gray", extent=EXT, origin="upper")
axes[0].set_title("Hillshade (315°/45°)", fontsize=11)

im = axes[1].imshow(slope_deg, cmap="hot_r", extent=EXT, origin="upper", vmin=0, vmax=30)
axes[1].set_title("Slope (degrees)", fontsize=11)
plt.colorbar(im, ax=axes[1], label="slope (°)", shrink=0.7)

for ax in axes:
    ax.set_xlabel("Easting (m)", fontsize=10)
axes[0].set_ylabel("Northing (m)", fontsize=10)
plt.tight_layout()
plt.show()

> **Inspect the DEM:**
> 1. The San Andreas fault crosses this scene from NW to SE. Identify the fault trace on the hillshade. What topographic expressions indicate the fault zone?
> 2. Wallace Creek enters the scene from the NE (upper right). Trace its path across the image. Where does it deflect, and in which direction? What does the deflection direction tell you about the sense of fault slip?
> 3. Look for older, abandoned channel traces on the **SW side** of the fault (lower left). How many can you identify?
> 4. How does the slope map complement the hillshade for identifying tectonic features?

## 3. Define the fault coordinate system

To measure fault-parallel offsets we rotate from UTM (Easting, Northing) coordinates into a fault-aligned system where:
- $x_\parallel$ = distance along the fault (positive NW)
- $x_\perp$ = distance perpendicular to the fault (positive NE, toward the hills)

The San Andreas at Wallace Creek strikes approximately **N45°W (315°)**.

In [ ]:
# ── Fault geometry ────────────────────────────────────────────────────────────
FAULT_STRIKE_DEG = 315.0          # N45°W
FAULT_ORIGIN_E   = 242635.0       # UTM Easting of a point on the fault trace (m)
FAULT_ORIGIN_N   = 3906960.0      # UTM Northing

strike = np.deg2rad(FAULT_STRIKE_DEG)
# Unit vector along fault (NW-positive)
s_hat = np.array([-np.sin(np.deg2rad(180-FAULT_STRIKE_DEG)),
                   np.cos(np.deg2rad(180-FAULT_STRIKE_DEG))])
# Simpler: fault strikes NW = decreasing E, increasing N
s_hat = np.array([-np.sin(np.deg2rad(45)), np.cos(np.deg2rad(45))])  # (-0.707, 0.707)
# Unit vector perpendicular to fault (NE-positive, toward hills)
n_hat = np.array([np.cos(np.deg2rad(45)), np.sin(np.deg2rad(45))])   # (0.707, 0.707)

print(f"Fault strike: {FAULT_STRIKE_DEG}° (N45°W)")
print(f"Along-fault unit vector (NW): {s_hat}")
print(f"Fault-normal unit vector (NE): {n_hat}")

def utm_to_fault(E, N):
    """Convert UTM (E, N) to fault-parallel / fault-perpendicular coordinates.
    Returns (x_parallel_m, x_perp_m) both in metres.
    x_parallel > 0 is NW along the fault.
    x_perp    > 0 is NE (toward the Temblor Range).
    """
    dE = E - FAULT_ORIGIN_E
    dN = N - FAULT_ORIGIN_N
    x_par  = dE * s_hat[0] + dN * s_hat[1]
    x_perp = dE * n_hat[0] + dN * n_hat[1]
    return x_par, x_perp

In [ ]:
# ── Annotated overview map ────────────────────────────────────────────────────
# Draw the fault trace across the DEM
t_vals   = np.linspace(-500, 500, 100)  # metres along fault
fault_E  = FAULT_ORIGIN_E + t_vals * s_hat[0]
fault_N  = FAULT_ORIGIN_N + t_vals * s_hat[1]

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(hs, cmap="gray", extent=EXT, origin="upper", vmin=0, vmax=1)
ax.plot(fault_E, fault_N, "r-", lw=1.5, alpha=0.8, label="SAF trace (approx.)")

# Mark the fault origin point
ax.plot(FAULT_ORIGIN_E, FAULT_ORIGIN_N, "r+", ms=14, mew=2)

# Compass
ax.annotate("", xy=(XLL+870, YLL+120), xytext=(XLL+870, YLL+40),
            arrowprops=dict(arrowstyle="->", color="k", lw=2))
ax.text(XLL+870, YLL+130, "N", ha="center", fontsize=12, fontweight="bold")

# Scale bar
ax.plot([XLL+50, XLL+150], [YLL+30, YLL+30], "k-", lw=3)
ax.text(XLL+100, YLL+50, "100 m", ha="center", fontsize=9)

ax.set_xlabel("Easting (m, UTM Zone 11)", fontsize=11)
ax.set_ylabel("Northing (m)", fontsize=11)
ax.set_title("Wallace Creek — fault trace and channel system", fontsize=12)
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.show()

## 4. Measure channel offsets

The offset of each channel is measured as the **fault-parallel distance** between the projected intersection of the channel walls on each side of the fault. We extract topographic profiles parallel to the fault on both sides and identify the channel margins from slope breaks.

### Method

For each channel we:
1. Draw a swath profile parallel to the fault at a fixed fault-perpendicular distance on each side
2. Plot elevation and slope along that profile
3. Identify the fault-parallel position of the channel walls from slope breaks
4. The offset = difference in wall positions between the SW and NE sides

In [ ]:
def extract_profile(dem_grid, east_arr, north_arr, E0, N0, along_hat, n_points=400, half_width_m=600):
    """Extract a swath profile along a line through (E0, N0) in direction along_hat.
    Returns (along_distance_m, median_elevation, std_elevation) arrays.
    """
    # Sample points along the profile
    t = np.linspace(-half_width_m, half_width_m, n_points)
    E_pts = E0 + t * along_hat[0]
    N_pts = N0 + t * along_hat[1]

    # Bilinear interpolation
    dE = east_arr[1] - east_arr[0]
    dN = north_arr[0] - north_arr[1]  # north_arr is decreasing with row index

    col_f = (E_pts - east_arr[0]) / dE
    row_f = (north_arr[0] - N_pts) / dN

    col_i = np.clip(col_f.astype(int), 0, dem_grid.shape[1]-2)
    row_i = np.clip(row_f.astype(int), 0, dem_grid.shape[0]-2)

    fc = col_f - col_i
    fr = row_f - row_i

    elev = (
        dem_grid[row_i,   col_i  ] * (1-fc) * (1-fr)
      + dem_grid[row_i,   col_i+1] *  fc    * (1-fr)
      + dem_grid[row_i+1, col_i  ] * (1-fc) *  fr
      + dem_grid[row_i+1, col_i+1] *  fc    *  fr
    )
    elev[elev == NODATA] = np.nan
    return t, elev

In [ ]:
# ── Active channel ────────────────────────────────────────────────────────────
# Extract fault-parallel profiles on each side of the fault
# NE side (upslope, toward hills): x_perp ≈ +30 m
# SW side (downslope): x_perp ≈ -30 m

PERP_OFFSET = 30.0   # metres from fault trace for each profile

# Profile origin points (shifted perpendicular to fault)
NE_origin = (FAULT_ORIGIN_E + PERP_OFFSET * n_hat[0],
             FAULT_ORIGIN_N + PERP_OFFSET * n_hat[1])
SW_origin = (FAULT_ORIGIN_E - PERP_OFFSET * n_hat[0],
             FAULT_ORIGIN_N - PERP_OFFSET * n_hat[1])

t_NE, elev_NE = extract_profile(dem, east, north, *NE_origin, s_hat, half_width_m=450)
t_SW, elev_SW = extract_profile(dem, east, north, *SW_origin, s_hat, half_width_m=450)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Overview map with profile locations
ax = axes[0, 0]
ax.imshow(hs, cmap="gray", extent=EXT, origin="upper", vmin=0, vmax=1)
ax.plot(fault_E, fault_N, "r-", lw=1.5, alpha=0.8)
t_line = np.linspace(-450, 450, 100)
ax.plot(NE_origin[0] + t_line*s_hat[0], NE_origin[1] + t_line*s_hat[1],
        "b--", lw=1.2, label="NE profile (+30 m)")
ax.plot(SW_origin[0] + t_line*s_hat[0], SW_origin[1] + t_line*s_hat[1],
        "g--", lw=1.2, label="SW profile (-30 m)")
ax.set_title("Profile locations", fontsize=10)
ax.legend(fontsize=8, frameon=False)
ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")

# NE profile
ax = axes[0, 1]
ax.plot(t_NE, elev_NE, "b-", lw=1.5)
ax.set_xlabel("Fault-parallel distance (m, +NW)")
ax.set_ylabel("Elevation (m)")
ax.set_title("NE side — fault-parallel profile")
ax.grid(True, alpha=0.3)

# SW profile
ax = axes[1, 1]
ax.plot(t_SW, elev_SW, "g-", lw=1.5)
ax.set_xlabel("Fault-parallel distance (m, +NW)")
ax.set_ylabel("Elevation (m)")
ax.set_title("SW side — fault-parallel profile")
ax.grid(True, alpha=0.3)

# Overlay comparison
ax = axes[1, 0]
ax.plot(t_NE, elev_NE - np.nanmean(elev_NE), "b-", lw=1.5, label="NE side")
ax.plot(t_SW, elev_SW - np.nanmean(elev_SW), "g-", lw=1.5, label="SW side")
ax.set_xlabel("Fault-parallel distance (m, +NW)")
ax.set_ylabel("Relative elevation (m)")
ax.set_title("Comparison (mean-removed)")
ax.legend(fontsize=9, frameon=False)
ax.grid(True, alpha=0.3)

plt.suptitle("Profiles parallel to the SAF — active channel region", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Offset measurement tool ───────────────────────────────────────────────────
# Students identify channel wall positions from the profiles above and enter
# them here. The along-fault coordinate t=0 corresponds to the fault origin.
#
# For each channel, identify the fault-parallel position of the channel wall
# on the NE side and the SW side. The offset is SW_position - NE_position
# (positive = NW displacement of SW block = right-lateral).

def measure_offset(NE_wall_m, SW_wall_m, label="channel"):
    """
    Compute right-lateral offset from channel wall positions in fault coordinates.
    NE_wall_m : fault-parallel position of channel wall on NE (uphill) side
    SW_wall_m : fault-parallel position of matching wall on SW (downhill) side
    Returns offset in metres (positive = right-lateral / NW displacement).
    """
    offset = SW_wall_m - NE_wall_m
    print(f"{label:30s}  NE wall: {NE_wall_m:+7.1f} m   SW wall: {SW_wall_m:+7.1f} m   "
          f"Offset: {offset:.1f} m")
    return offset

print("\n*** FILL IN YOUR MEASUREMENTS FROM THE PROFILES ABOVE ***\n")
print(f"{'Channel':30s}  {'NE wall':>10} {'SW wall':>10} {'Offset':>10}")
print("-" * 65)

# ──────────────────────────────────────────────────────────────────────────────
# EDIT THESE VALUES after inspecting the profiles
# Hint: look for channel-wall breaks (sudden slope change) in the elevation profiles.
# Channel walls appear as sharp elevation steps.
# ──────────────────────────────────────────────────────────────────────────────
active_NE_SE_wall  =  -70.0   # fault-parallel position of SE (right) channel wall, NE side
active_SW_SE_wall  =   58.0   # same wall, SW side (offset NW by slip)

active_NE_NW_wall  = -140.0   # NW (left) channel wall, NE side
active_SW_NW_wall  =  -12.0   # NW channel wall, SW side

off_SE = measure_offset(active_NE_SE_wall, active_SW_SE_wall, "Active — SE wall")
off_NW = measure_offset(active_NE_NW_wall, active_SW_NW_wall, "Active — NW wall")

ACTIVE_OFFSET_M   = (off_SE + off_NW) / 2
ACTIVE_OFFSET_ERR = abs(off_SE - off_NW) / 2
print(f"\nBest estimate:  {ACTIVE_OFFSET_M:.1f} ± {ACTIVE_OFFSET_ERR:.1f} m")
print(f"Sieh & Jahns (1984):  128 ± 1 m")

In [ ]:
# ── Zoomed maps for all three channels ───────────────────────────────────────
# Channel crossing locations (approximate, UTM Zone 11)
# Adjust these based on your inspection of the hillshade
CHANNELS = {
    "Active":         {"E": 242660, "N": 3906980, "offset_lit": 128,  "err_lit": 1},
    "2nd abandoned":  {"E": 242700, "N": 3906900, "offset_lit": 370,  "err_lit": 10},
    "1st abandoned":  {"E": 242740, "N": 3906820, "offset_lit": 475,  "err_lit": 3},
}

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=False)

for ax, (name, ch) in zip(axes, CHANNELS.items()):
    # Zoom window: ±200 m around each channel crossing
    hw = 200   # half-width in metres
    zoom_ext = [ch["E"]-hw, ch["E"]+hw, ch["N"]-hw, ch["N"]+hw]

    # Crop DEM and hillshade to zoom window
    col_sl = slice(int((zoom_ext[0]-XLL)/CELL), int((zoom_ext[1]-XLL)/CELL))
    row_sl = slice(int((YLL+NROWS*CELL-zoom_ext[3])/CELL),
                   int((YLL+NROWS*CELL-zoom_ext[2])/CELL))
    hs_crop = hillshade(dem[row_sl, col_sl], CELL)

    ax.imshow(hs_crop, cmap="gray", extent=zoom_ext, origin="upper")

    # Draw fault trace
    t_z = np.linspace(-250, 250, 100)
    ax.plot(ch["E"] + t_z*s_hat[0], ch["N"] + t_z*s_hat[1],
            "r-", lw=1.5, alpha=0.8)

    # Arrow showing offset direction
    ax.annotate("", xy=(ch["E"] + 30*s_hat[0] + 10, ch["N"] + 30*s_hat[1]),
                xytext=(ch["E"] + 10, ch["N"]),
                arrowprops=dict(arrowstyle="->", color="orange", lw=1.5))

    ax.set_title(f"{name}\n(lit. offset: {ch['offset_lit']} ± {ch['err_lit']} m)",
                 fontsize=10)
    ax.set_xlabel("Easting (m)")
    ax.tick_params(labelsize=8)

axes[0].set_ylabel("Northing (m)")
plt.suptitle("Zoomed views of three Wallace Creek channels at the SAF", fontsize=12)
plt.tight_layout()
plt.show()

> **Measure the offsets:**
> 1. Using the zoomed maps and the profile tool above, estimate the fault-parallel offset of each channel. Fill in the measurement cells below for the 2nd abandoned and 1st abandoned channels by analogy with the active channel example.
> 2. Sieh and Jahns (1984) report the active channel offset as 128 ± 1 m. Is this error estimate realistic given the uncertainties you encountered?
> 3. Which channel wall (NW or SE) is easier to measure, and why?
> 4. The 1st abandoned channel is the most subtle feature in the LiDAR. Why would it be harder to see, and what geomorphic processes might have degraded the original channel form?

In [ ]:
# ── Fill in your measurements for all three channels ─────────────────────────
# Structure: (offset_m, uncertainty_m)
# Replace the literature values with your own measurements

# ⬇⬇ EDIT THESE ⬇⬇
MY_OFFSETS = {
    "Active":        (128,  5),   # (your measurement in m, your uncertainty in m)
    "2nd abandoned": (370, 20),
    "1st abandoned": (475, 25),
}

print(f"{'Channel':20s}  {'Offset (m)':>12}  {'Your uncertainty (m)':>22}  {'Lit. (Sieh & Jahns)':>22}")
print("-" * 82)
lit = {"Active": (128,1), "2nd abandoned": (370,10), "1st abandoned": (475,3)}
for name, (off, err) in MY_OFFSETS.items():
    lo, le = lit[name]
    print(f"{name:20s}  {off:>12.0f}  {err:>22.0f}  {lo} ± {le}")

## 5. Interpret the geomorphic history

Sieh and Jahns (1984) collected radiocarbon dates from sediments deposited in each channel. Use these dates and the cross-cutting relationships you observe in the LiDAR to reconstruct the timeline of channel avulsions.

| Unit | Description | $^{14}$C age (ka) | Relates to channel |
|---|---|---|---|
| **Hl** | Active channel alluvium | — | Active channel |
| **Hh** top | High channel fill | 3.8 ± 0.2 | 2nd abandoned (younger) |
| **Hh** base | High channel fill | 5.8 ± 0.2 | 2nd abandoned (younger) |
| **Py fan** base | Younger Late Pleistocene fan | 13.3 ± 1.6 | 2nd abandoned (younger) |
| **Po fan** | Older Late Pleistocene fan | 19.3 ± 1.0 | 1st abandoned (older) |

In [ ]:
# ── Channel age estimates ─────────────────────────────────────────────────────
# For each channel we need an age for when it was ACTIVE at the fault crossing.
# This is when slip offset was zero — i.e. when the channel was first created.
#
# The age of the channel = age of avulsion that created it.
# Once a new channel forms, the old one stops accumulating slip.
#
# Think about: which C-14 age constrains the avulsion that CREATED each channel?
#              which constrains when the channel was ABANDONED?

# ⬇⬇ EDIT THESE — enter your best age estimate and 1-sigma uncertainty ⬇⬇
AGES_KA = {
    # (age_ka, sigma_ka)  — age = time of avulsion (zero slip)
    "Active":        (3.7,  0.2),   # Hh top, ~3.7-3.8 ka
    "2nd abandoned": (13.3, 1.6),   # Py fan base (maximum age of avulsion)
    "1st abandoned": (19.3, 1.0),   # Po fan age
}

print("Age estimates for channel avulsions:")
print(f"{'Channel':20s}  {'Age (ka)':>10}  {'1-sigma (ka)':>14}  {'Constraint'}")
print("-" * 75)
constraints = {
    "Active":        "Hh top (~3.7-3.8 ka)",
    "2nd abandoned": "Py fan base (13.3 ± 1.6 ka)",
    "1st abandoned": "Po fan (19.3 ± 1.0 ka)",
}
for name, (age, sig) in AGES_KA.items():
    print(f"{name:20s}  {age:>10.1f}  {sig:>14.1f}  {constraints[name]}")

print("\nNote: ages are in ka (thousands of years). Uncertainties are 1-sigma.")

> **Interpret the geomorphic history:**
> 1. Place the three avulsion events in chronological order. What triggered each avulsion — gradual deflection, a single flood, or something else?
> 2. The Hh deposits have two ages (3.7-3.8 ka top, 5.8 ± 0.2 ka base). What does this range tell you about when the second abandoned channel was active?
> 3. The 1st abandoned channel is not preserved NE of the fault. What does that tell you about the age of the deposits on the NE side?
> 4. Why does slip rate estimation using the active channel give a minimum slip rate, while the older channels might give closer to the long-term average?

## 6. Calculate slip rates

The slip rate is simply
$$
\dot{s} = \frac{\text{offset}}{\text{age}}
$$
but both offset and age carry uncertainty. We propagate uncertainties using a **Monte Carlo approach**: draw many random samples of offset and age from their probability distributions, compute the slip rate for each sample, and report the distribution of the results.

In [ ]:
rng = np.random.default_rng(42)
N_SAMPLES = 100_000

results = {}
for name in MY_OFFSETS:
    off_m, off_err = MY_OFFSETS[name]
    age_ka, age_err = AGES_KA[name]

    # Draw samples (assume Gaussian)
    off_samples = rng.normal(off_m,   off_err,  N_SAMPLES)  # metres
    age_samples = rng.normal(age_ka,  age_err,  N_SAMPLES)  # ka

    # Keep only physically plausible draws
    valid = (off_samples > 0) & (age_samples > 0)
    rate_samples = off_samples[valid] / (age_samples[valid] * 1e3)  # m / yr → mm/yr

    results[name] = {
        "rate_samples": rate_samples,
        "median":  np.median(rate_samples),
        "p16":     np.percentile(rate_samples, 16),
        "p84":     np.percentile(rate_samples, 84),
        "p2p5":    np.percentile(rate_samples, 2.5),
        "p97p5":   np.percentile(rate_samples, 97.5),
    }

print(f"{'Channel':20s}  {'Slip rate':>12}  {'1σ range':>18}  {'2σ range':>18}")
print("-" * 75)
for name, r in results.items():
    print(f"{name:20s}  {r['median']:>9.1f} mm/yr  "
          f"[{r['p16']:.1f}, {r['p84']:.1f}] mm/yr  "
          f"[{r['p2p5']:.1f}, {r['p97p5']:.1f}] mm/yr")

print(f"\nSieh & Jahns (1984): 33.9 mm/yr")

In [ ]:
# ── Visualize slip rate distributions ─────────────────────────────────────────
colors = {"Active": "#2a78d6", "2nd abandoned": "#6250d6", "1st abandoned": "#eb6834"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)

for ax, (name, r) in zip(axes, results.items()):
    col = colors[name]
    ax.hist(r["rate_samples"], bins=80, color=col, alpha=0.7, density=True)
    ax.axvline(r["median"], color=col, lw=2, label=f"Median: {r['median']:.1f} mm/yr")
    ax.axvspan(r["p16"], r["p84"], alpha=0.2, color=col, label="68% CI")
    ax.axvline(33.9, color="k", lw=1.5, ls="--", label="Sieh & Jahns: 33.9 mm/yr")
    ax.set_xlabel("Slip rate (mm/yr)", fontsize=10)
    ax.set_ylabel("Probability density", fontsize=10)
    ax.set_title(f"{name}\noffset: {MY_OFFSETS[name][0]} ± {MY_OFFSETS[name][1]} m  "
                 f"age: {AGES_KA[name][0]} ± {AGES_KA[name][1]} ka", fontsize=9)
    ax.legend(fontsize=8, frameon=False)

plt.suptitle("Monte Carlo slip rate distributions — Wallace Creek, SAF", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cumulative offset–age plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, col in colors.items():
    off_m, off_err = MY_OFFSETS[name]
    age_ka, age_err = AGES_KA[name]
    ax.errorbar(age_ka, off_m,
                xerr=age_err, yerr=off_err,
                fmt="o", color=col, ms=9, capsize=5, label=name)

# Best-fit line through origin
all_ages = np.array([v[0] for v in AGES_KA.values()]) * 1e3   # yr
all_offs = np.array([v[0] for v in MY_OFFSETS.values()])
rate_best = np.sum(all_offs * all_ages) / np.sum(all_ages**2)  # weighted by age

t_fit = np.linspace(0, 22, 100)
ax.plot(t_fit, rate_best * t_fit * 1e3 / 1e3, "k--", lw=1.5,
        label=f"Best fit: {rate_best*1e3:.1f} mm/yr")

# Sieh & Jahns line
ax.plot(t_fit, 33.9e-3 * t_fit * 1e3, "gray", lw=1.5, ls=":",
        label="Sieh & Jahns: 33.9 mm/yr")

ax.set_xlabel("Age (ka)", fontsize=11)
ax.set_ylabel("Cumulative offset (m)", fontsize=11)
ax.set_title("Offset vs. age — Wallace Creek", fontsize=12)
ax.legend(fontsize=10, frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 22)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

print(f"Weighted best-fit rate: {rate_best*1e3:.1f} mm/yr")

## Synthesis

> **Answer each question in one or two sentences:**
> 1. Do your three slip rate estimates agree with each other within uncertainty? If not, what might explain the discrepancy?
> 2. What are the three largest sources of uncertainty in a geomorphic slip rate? Which is hardest to quantify?
> 3. If the SAF has been slipping at ~34 mm/yr for the full interseismic period (~100–150 yr since the last great earthquake), what slip deficit has accumulated on this segment? What magnitude earthquake would release it (use your moment budget from Lab 4)?
> 4. The Sieh and Jahns (1984) study was published 40 years ago and is still widely cited. What additional data or methods would a modern study bring to bear on this problem?
> 5. Explain why the offset of the active channel gives a **minimum** bound on slip rate rather than the long-term average.

## Summary

- The San Andreas fault at Wallace Creek slips at approximately **34 mm/yr** based on three generations of offset channels spanning 3.7–19 ka.
- Fault offsets are measured by projecting channel walls to the fault trace and computing the fault-parallel displacement.
- The dominant uncertainties are in the age dating (radiocarbon), the offset measurement (projection assumptions), and the timing of channel avulsion relative to the dated deposits.
- Monte Carlo uncertainty propagation accounts for the non-symmetric probability distributions that arise when dividing two uncertain quantities.
- The offset–age relationship appears approximately linear, consistent with a roughly constant slip rate over the Holocene.